![pageindex_banner](https://pageindex.ai/static/images/pageindex_banner.jpg)

<p align="center"><i>Reasoning-based RAG&nbsp; ✧ &nbsp;No Vector DB&nbsp; ✧ &nbsp;No Chunking&nbsp; ✧ &nbsp;Human-like Retrieval</i></p>

<p align="center">
  <a href="https://vectify.ai">🏠 Homepage</a>&nbsp; • &nbsp;
  <a href="https://dash.pageindex.ai">🖥️ Dashboard</a>&nbsp; • &nbsp;
  <a href="https://docs.pageindex.ai/quickstart">📚 API Docs</a>&nbsp; • &nbsp;
  <a href="https://github.com/VectifyAI/PageIndex">📦 GitHub</a>&nbsp; • &nbsp;
  <a href="https://discord.com/invite/VuXuf29EUj">💬 Discord</a>&nbsp; • &nbsp;
  <a href="https://ii2abc2jejf.typeform.com/to/tK3AXl8T">✉️ Contact</a>&nbsp;
</p>

<div align="center">

[![Star us on GitHub](https://img.shields.io/github/stars/VectifyAI/PageIndex?style=for-the-badge&logo=github&label=⭐️%20Star%20Us)](https://github.com/VectifyAI/PageIndex) &nbsp;&nbsp; [![Follow us on X](https://img.shields.io/badge/Follow%20Us-000000?style=for-the-badge&logo=x&logoColor=white)](https://twitter.com/VectifyAI)

</div>

---

# Simple Vectorless RAG with PageIndex

## PageIndex Introduction
PageIndex is a new **reasoning-based**, **vectorless RAG** framework that performs retrieval in two steps:  
1. Generate a tree structure index of documents  
2. Perform reasoning-based retrieval through tree search  

<div align="center">
  <img src="https://docs.pageindex.ai/images/cookbook/vectorless-rag.png" width="70%">
</div>

Compared to traditional vector-based RAG, PageIndex features:
- **No Vectors Needed**: Uses document structure and LLM reasoning for retrieval.
- **No Chunking Needed**: Documents are organized into natural sections rather than artificial chunks.
- **Human-like Retrieval**: Simulates how human experts navigate and extract knowledge from complex documents. 
- **Transparent Retrieval Process**: Retrieval based on reasoning — say goodbye to approximate semantic search ("vibe retrieval").

## 📝 Notebook Overview

This notebook demonstrates a simple, minimal example of **vectorless RAG** with PageIndex. You will learn how to:
- [x] Build a PageIndex tree structure of a document
- [x] Perform reasoning-based retrieval with tree search
- [x] Generate answers based on the retrieved context

> ⚡ Note: This is a **minimal example** to illustrate PageIndex's core philosophy and idea, not its full capabilities. More advanced examples are coming soon.

---

## Step 0: Preparation



#### 0.1 Install PageIndex

In [ ]:
%pip install -q --upgrade pageindex boto3

#### 0.2 Setup PageIndex (Local)

In [1]:
import sys
import textwrap
import concurrent.futures

# Remove any cached pageindex modules so the local repo version takes precedence
for key in list(sys.modules.keys()):
    if 'pageindex' in key:
        del sys.modules[key]

sys.path.insert(0, 'c:/Users/vrang/code/PageIndex')

from pageindex import page_index
import pageindex.utils as utils

# Run page_index in a thread so asyncio.run() gets a fresh event loop
# (avoids conflicts with Jupyter's running loop on Python 3.14)
def run_page_index(*args, **kwargs):
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(page_index, *args, **kwargs).result()

# Helper: create flat mapping of node_id → node
def create_node_mapping(tree):
    mapping = {}
    def traverse(nodes):
        for node in nodes:
            if 'node_id' in node:
                mapping[node['node_id']] = node
            if 'nodes' in node:
                traverse(node['nodes'])
    traverse(tree)
    return mapping

# Helper: print text with line wrapping
def print_wrapped(text, width=100):
    for line in text.split('\n'):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()

#### 0.3 Setup LLM

We use AWS Bedrock's Nova Lite model via litellm for both tree generation and reasoning-based retrieval. Make sure your AWS credentials are configured (e.g. via `aws configure` or environment variables).

In [2]:
import os
import json
import asyncio
from litellm import completion

# Use sync litellm.completion run in a thread executor to avoid
# aiohttp "Timeout should be used inside a task" on Python 3.14
async def call_llm(prompt, model="bedrock/us.amazon.nova-2-lite-v1:0", temperature=0):
    loop = asyncio.get_event_loop()
    response = await loop.run_in_executor(
        None,
        lambda: completion(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
        )
    )
    return response.choices[0].message.content.strip()

## Step 1: PageIndex Tree Generation

#### 1.1 Generate PageIndex tree from a local PDF

In [14]:
import os, json

pdf_path = r"C:\Users\vrang\code\PageIndex\tests\pdfs\tcs_annual-report-2024-2025-trunc.pdf"

# Generate tree structure locally (no API required)
result = run_page_index(
    doc=pdf_path,
    model="bedrock/us.amazon.nova-2-lite-v1:0",
    if_add_node_id="yes",
    if_add_node_summary="yes",
    if_add_node_text="yes",
)
tree = result["structure"]

# Save results
pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
output_dir = '../results'
output_file = f'{output_dir}/{pdf_name}_structure.json'
os.makedirs(output_dir, exist_ok=True)
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
print(f'Tree structure saved to: {output_file}')
print("Tree structure generated successfully!")

Parsing PDF...
start find_toc_pages
toc found
start detect_page_index
index found
process_toc_with_page_numbers
start_index: 1
start toc_transformer
start toc_index_extractor
Document validation: 50 pages, max allowed index: 50
Truncated 23 TOC items that exceeded document length
start verify_toc
check all items
accuracy: 33.33%
process_toc_no_page_numbers
start_index: 1
start toc_transformer
divide page_list to groups 2
Document validation: 50 pages, max allowed index: 50
start verify_toc
check all items
accuracy: 100.00%
Tree structure saved to: ../results/tcs_annual-report-2024-2025-trunc_structure.json
Tree structure generated successfully!


#### 1.2 Inspect the generated PageIndex tree structure

In [8]:
print('Tree Structure of the Document:')
utils.print_toc(tree)

Tree Structure of the Document:
Preface
Notice
Customer Stories
  TCS’ AI Powered Drug Design Enables Pharma’s Fight Against Cancer
  Revolutionizing Employee Experience with GenAI
  Transforming Telecommunications with GenAI
  Digital Transformation of Electoral Roll Management in India
  A Journey Towards Becoming the Most Trusted Advisor
  AI - The Game Changer for Logistics and Supply Chain
  Transforming Airline Operations with an AI-equipped, Unified Data Platform
  One step closer to a Net-Zero Grid
  GenAI Revolutionizing Media Content Analysis
  Re-imagining Banking with AI


## Step 2: Reasoning-Based Retrieval with Tree Search

#### 2.1 Use LLM for tree search and identify nodes that might contain relevant context

In [9]:
import json

query = "What is the company doing on AI?"

tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result = await call_llm(search_prompt, model="bedrock/us.amazon.nova-2-lite-v1:0")

#### 2.2 Print retrieved nodes and reasoning process

In [10]:
node_map = create_node_mapping(tree)
tree_search_result_json = utils.extract_json(tree_search_result)

print('Reasoning Process:')
print_wrapped(tree_search_result_json['thinking'])

print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Start Page: {node.get('start_index', 'N/A')}\t Title: {node['title']}")

Reasoning Process:
The question asks about what the company is doing on AI. The document is an Integrated Annual Report
2024-25 for Tata Consultancy Services (TCS). Looking through the nodes:

1. Node 0000 (Preface) mentions TCS's focus on AI/GenAI-first learning in human capital and lists
AI-related focus areas in intellectual capital: Purposeful AI, Computing Futures, Digital Sciences.
It also mentions partnerships with NVIDIA, Google Cloud, etc., and customer success stories
involving AI.

2. Node 0003 (TCS’AI Powered Drug Design Enables Pharma’s Fight Against Cancer) details a specific
AI application in pharmaceuticals for cancer drug design.

3. Node 0004 (Revolutionizing Employee Experience with GenAI) discusses GenAI applications in
employee experience and business operations for Proximus.

4. Node 0005 (Transforming Telecommunications with GenAI) covers GenAI use in telecommunications for
Proximus.

5. Node 0008 (AI - The Game Changer for Logistics and Supply Chain) discusses A

## Step 3: Answer Generation

#### 3.1 Extract relevant context from retrieved nodes

In [11]:
node_list = utils.extract_json(tree_search_result)["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
print_wrapped(relevant_content[:1000] + '...')

Retrieved Context:

Integrated Annual Report 2024-25
2024-25Integrated Annual Report
03 March 1839 – 19 May 1904Jamsetji Nusserwanji TataOur Founder
In a free enterprise, the
community is not just another
stakeholder in business, but is in
fact the very purpose of
its existenceRemembering Mr. T ata
Padma Vibhushan
Ratan N Tata
28 December 1937 – 9 October 2024
It is with a profound sense of loss that we bid farewell
to Mr. Ratan Naval Tata, a truly uncommon leader
whose immeasurable contributions have shaped
not only the Tata Group but also the very fabric of our
nation.
For the Tata Group, Mr. Tata was more than a
chairperson. He inspired by example. With an
unwavering commitment to excellence, integrity and
innovation, the Tata Group under his stewardship
expanded its global footprint while always remaining
true to its moral compass.Mr. Tata’s dedication to philanthropy and the
development of society has touched the lives of
millions. From education to healthcare, his initiatives
hav

#### 3.2 Generate answer based on retrieved context

In [12]:
answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

print('Generated Answer:\n')
answer = await call_llm(answer_prompt, model="bedrock/us.amazon.nova-2-lite-v1:0")
print_wrapped(answer)

Generated Answer:

TCS is proactively leading the transition to AI by systematically infusing AI across its offerings
and building intelligent agent solutions throughout the value chain. They plan to deliver solutions
through a human+AI model, invest in AI infrastructure, and forge industry-best partnerships.


---

## 🎯 What's Next

This notebook has demonstrated a **basic**, **minimal** example of **reasoning-based**, **vectorless** RAG with PageIndex. The workflow illustrates the core idea:
> *Generating a hierarchical tree structure from a document, reasoning over that tree structure, and extracting relevant context, without relying on a vector database or top-k similarity search*.

While this notebook highlights a minimal workflow, the PageIndex framework is built to support **far more advanced** use cases. In upcoming tutorials, we will introduce:
* **Multi-Node Reasoning with Content Extraction** — Scale tree search to extract and select relevant content from multiple nodes.
* **Multi-Document Search** — Enable reasoning-based navigation across large document collections, extending beyond a single file.
* **Efficient Tree Search** — Improve tree search efficiency for long documents with a large number of nodes.
* **Expert Knowledge Integration and Preference Alignment** — Incorporate user preferences or expert insights by adding knowledge directly into the LLM tree search, without the need for fine-tuning.



## 🔎 Learn More About PageIndex
  <a href="https://vectify.ai">🏠 Homepage</a>&nbsp; • &nbsp;
  <a href="https://dash.pageindex.ai">🖥️ Dashboard</a>&nbsp; • &nbsp;
  <a href="https://docs.pageindex.ai/quickstart">📚 API Docs</a>&nbsp; • &nbsp;
  <a href="https://github.com/VectifyAI/PageIndex">📦 GitHub</a>&nbsp; • &nbsp;
  <a href="https://discord.com/invite/VuXuf29EUj">💬 Discord</a>&nbsp; • &nbsp;
  <a href="https://ii2abc2jejf.typeform.com/to/tK3AXl8T">✉️ Contact</a>

<br>

© 2025 [Vectify AI](https://vectify.ai)